# 过滤 Think 标签后重新计算 ROUGE 评估指标

## 一、导入依赖

In [1]:
import re                          # 标准库：正则表达式，用于匹配并去除 <think>...</think> 标签
import json                        # 标准库：解析 .jsonl 文件中的每一行 JSON 对象
from rouge_chinese import Rouge    # 第三方库：专为中文设计的 ROUGE 实现，内部集成 jieba 分词
import jieba                       # 第三方库：中文分词，rouge_chinese 依赖此库对文本预处理

## 二、配置参数

In [2]:
# LlamaFactory do_predict 生成的预测结果文件路径（每行一个 JSON 对象）
PREDICTIONS_PATH_BeforeTrain = "./output/predict_results_BeforeTrain/generated_predictions.jsonl"
PREDICTIONS_PATH_AfterTrain = "./output/predict_results_AfterTrain/generated_predictions.jsonl"
# 匹配 <think>...</think> 的正则表达式（re.DOTALL 使 . 能匹配换行符，处理多行推理链）
THINK_PATTERN = re.compile(r"<think>.*?</think>", re.DOTALL)

## 三、数据加载与预处理

### 3.1 加载预测结果文件

In [3]:
def load_jsonl(path: str) -> list[dict]:
    """
    按行读取 JSONL 文件，返回所有记录构成的列表。

    参数：
        path (str): JSONL 文件路径，每行为一个合法的 JSON 对象。

    返回：
        list[dict]: 每个元素对应文件中的一行记录，
                    LlamaFactory 格式下每条记录包含 'predict' 和 'label' 两个字段。
    """
    records = []                                    # 初始化空列表，存储所有解析后的记录
    with open(path, "r", encoding="utf-8") as f:   # 以 UTF-8 编码打开文件（中文内容必须指定编码）
        for line in f:                              # 逐行读取，每行为一个独立的 JSON 字符串
            line = line.strip()                     # 去除行首行尾空白字符（包括换行符）
            if line:                                # 跳过空行，避免 json.loads 抛出异常
                records.append(json.loads(line))    # 将 JSON 字符串解析为 dict 并追加到列表
    return records                                  # 返回值：list[dict]，长度等于文件中有效行数

### 3.2 过滤 Think 标签并提取预测文本与参考标签

In [4]:
def strip_think(text: str) -> str:
    """
    去除文本中所有 <think>...</think> 推理链内容，保留最终答案部分。

    参数：
        text (str): 原始生成文本，可能包含一段或多段 <think>...</think> 推理链。

    返回：
        str: 去除全部推理链后经 strip() 处理的干净文本。
    """
    cleaned = THINK_PATTERN.sub("", text)   # 用空字符串替换所有匹配的 <think>...</think> 片段
    return cleaned.strip()                  # 去除替换后产生的首尾多余空白，返回干净的答案文本


def build_rouge_inputs(path: str) -> tuple[list[str], list[str], list[str]]:
    """
    读取 JSONL 文件，过滤 <think> 推理链，构建 ROUGE 评估所需的输入序列。

    参数：
        path (str): LlamaFactory generated_predictions.jsonl 文件路径。

    返回：
        tuple[list[str], list[str], list[str]]：
            preds_raw   - 原始预测文本列表（含 <think> 标签），供人工抽样审阅；list[str]
            preds_clean - 过滤推理链并经 jieba 分词后的预测文本列表；list[str]
            labels      - 经 jieba 分词后的参考答案列表；list[str]
    """
    records     = load_jsonl(path)   # 从文件加载所有记录；list[dict]
    preds_raw   = []                 # 原始预测文本（含 <think>），用于对比展示
    preds_clean = []                 # 去除推理链并分词后的预测文本，用于 ROUGE 计算
    labels      = []                 # 分词后的参考答案，与 preds_clean 一一对应

    for record in records:                                        # 遍历每条记录（dict）
        pred_raw   = record.get("predict", "")                    # 取 'predict' 字段，缺失时返回空串；str
        label      = record.get("label",   "")                    # 取 'label' 字段，缺失时返回空串；str
        pred_clean = strip_think(pred_raw)                        # 去除推理链；返回 str
        preds_raw.append(pred_raw)                                # 保留原始文本供抽样审阅
        preds_clean.append(" ".join(jieba.cut(pred_clean)))       # jieba 分词后空格拼接；rouge_chinese 要求
        labels.append(" ".join(jieba.cut(label)))                 # 参考答案同样做 jieba 分词处理

    print(f"[{path}] 加载完成，共 {len(records)} 条样本")         # 打印加载结果，确认文件读取正常
    return records, preds_raw, preds_clean, labels                # 同时返回 records 供抽样展示时访问原始标签

### 3.3 抽样展示过滤效果

In [5]:
SHOW_N = 3   # 每个文件各展示前 N 条样本，用于人工确认 <think> 过滤效果是否符合预期

# 构建两份数据（BeforeTrain 和 AfterTrain），同时加载并预处理
records_before, preds_raw_before, preds_clean_before, labels_before = build_rouge_inputs(PREDICTIONS_PATH_BeforeTrain)
records_after,  preds_raw_after,  preds_clean_after,  labels_after  = build_rouge_inputs(PREDICTIONS_PATH_AfterTrain)

# 遍历两组数据，分别抽样展示过滤效果
for tag, records, preds_raw, preds_clean in [
    ("训练前（BeforeTrain）", records_before, preds_raw_before, preds_clean_before),   # 第一组：训练前预测
    ("训练后（AfterTrain）",  records_after,  preds_raw_after,  preds_clean_after),    # 第二组：训练后预测
]:
    print(f"\n{'#'*60}")
    print(f"# 抽样展示 - {tag}")                               # 打印当前组别标题
    print(f"{'#'*60}")
    for i in range(min(SHOW_N, len(records))):                 # 最多取 SHOW_N 条，避免越界
        print(f"\n{'='*60}")
        print(f"[样本 {i+1}]")
        print(f"原始预测（前200字符）：\n{preds_raw[i][:200]}\n")    # 截取前200字符避免输出过长
        print(f"过滤后预测：\n{preds_clean[i]}\n")                   # 展示去除推理链后的干净预测文本
        print(f"参考标签：\n{records[i].get('label', '')}\n")        # 展示原始参考答案（未分词版本，便于阅读）

Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.447 seconds.
Prefix dict has been built successfully.


[./output/predict_results_BeforeTrain/generated_predictions.jsonl] 加载完成，共 1145 条样本
[./output/predict_results_AfterTrain/generated_predictions.jsonl] 加载完成，共 1145 条样本

############################################################
# 抽样展示 - 训练前（BeforeTrain）
############################################################

[样本 1]
原始预测（前200字符）：
</think><think>
嗯，我现在需要帮用户写一段广告文案，基于他们提供的商品属性标签。首先，我得仔细分析这些标签，看看这个衣服是什么样的。

标签里有“类型#上衣”，所以这是件衬衫。材质是“棉”，感觉很舒适，适合夏天穿着。“风格#街头”和“风格#休闲”结合起来，这件衬衫应该既有城市感，又不失休闲随意的感觉。

接下来，图案部分有三个选项：字母、文字和印花。这里可能需要综合考虑，或者选择一

过滤后预测：
< / think > 
 
 这是 一件 极具 魅力 的 街头 风格 休闲 卫衣 ， 采用 纯棉 面料 ， 柔软 透气 ， 穿着 舒适 。 卫衣 采用 抽绳 款式 设计 ， 搭配 连帽 造型 ， 显得 更加 时尚 大气 。 衬衫 上 印有 经典 印花 图案 ， 融合 字母 与 文字 元素 ， 展现 独特 的 艺术 感 。 无论是 日常 通勤 还是 周末 逛街 ， 这件 衬衫 都 能 让 你 轻松 成为 焦点 ！ 快 来 入手 这件 充满 街头 风采 的 棉质 卫衣 吧 ～ 👕 ✨ : / / : / / 
 
 < / think > 
 
 好 的 ， 我 明白 您 的 需求 是 希望 我 以 您 提供 的 商品 属性 标签 为 基础 ， 生成 一段 流畅 、 有 吸引力 的 中文 广告 文案 。 以下 是 我 对 您 需求 的 理解 和 思考 过程 ： 
 
 ###   分析 商品 属性 标签 ： 
 1 .   * * 类型 * * # 上衣     
 2 .   * * 材质 * *

## 四、计算 ROUGE 指标

In [6]:
rouge = Rouge()   # 实例化 rouge_chinese 的 Rouge 对象，默认计算 rouge-1 / rouge-2 / rouge-l


def evaluate_rouge(preds_clean: list[str], labels: list[str], tag: str) -> dict:
    """
    对过滤推理链后的预测序列与参考序列计算 ROUGE 指标。

    参数：
        preds_clean (list[str]): 经 jieba 分词、空格拼接的预测文本列表（已去除 <think> 标签）。
        labels      (list[str]): 经 jieba 分词、空格拼接的参考答案列表。
        tag         (str):       标识本次评估的名称（如 "训练前" / "训练后"），仅用于打印日志。

    返回：
        dict: rouge_chinese.get_scores 返回的均值字典，
              结构为 {'rouge-1': {'f':..,'p':..,'r':..}, 'rouge-2': {...}, 'rouge-l': {...}}。
    """
    # 过滤掉预测文本或标签为空的样本（空串会导致 rouge_chinese 抛出 ValueError）
    valid_pairs  = [(p, l) for p, l in zip(preds_clean, labels) if p.strip() and l.strip()]
    valid_preds  = [pair[0] for pair in valid_pairs]   # 有效预测列表；list[str]
    valid_labels = [pair[1] for pair in valid_pairs]   # 有效标签列表；list[str]
    skipped      = len(preds_clean) - len(valid_preds) # 被过滤的空样本数量；int

    print(f"[{tag}] 有效样本：{len(valid_preds)} / {len(preds_clean)}（{skipped} 条因空预测被过滤）")

    # avg=True：返回全部有效样本的均值 dict，而非逐条得分列表
    scores = rouge.get_scores(valid_preds, valid_labels, avg=True)
    return scores   # 返回值：dict，键为 'rouge-1'/'rouge-2'/'rouge-l'，值为含 f/p/r 的子字典


# 分别对训练前、训练后两份预测结果计算 ROUGE
scores_before = evaluate_rouge(preds_clean_before, labels_before, "训练前 BeforeTrain")  # dict
scores_after  = evaluate_rouge(preds_clean_after,  labels_after,  "训练后 AfterTrain")   # dict

[训练前 BeforeTrain] 有效样本：1145 / 1145（0 条因空预测被过滤）
[训练后 AfterTrain] 有效样本：1145 / 1145（0 条因空预测被过滤）


## 五、结果展示

In [7]:
def print_scores(scores: dict, tag: str) -> None:
    """
    格式化打印一组 ROUGE 评估结果。

    参数：
        scores (dict): evaluate_rouge 返回的均值字典，
                       结构为 {'rouge-1': {'f':..,'p':..,'r':..}, ...}。
        tag    (str):  标识本次评估的名称，打印在表头中。

    返回：
        None
    """
    print(f"\n{'='*55}")
    print(f"  过滤 <think> 后的 ROUGE 评估结果 — {tag}")  # 打印标题，区分训练前/后
    print(f"{'='*55}")
    for metric, values in scores.items():               # 遍历三个指标：rouge-1, rouge-2, rouge-l
        f = values["f"] * 100                           # F1 分数，乘以100转换为百分制；float
        p = values["p"] * 100                           # Precision（精确率）；float
        r = values["r"] * 100                           # Recall（召回率）；float
        print(f"  {metric.upper():10s}  F1={f:.4f}  P={p:.4f}  R={r:.4f}")  # 对齐格式化输出


# 分别打印训练前、训练后的完整 ROUGE 指标
print_scores(scores_before, "训练前 BeforeTrain")
print_scores(scores_after,  "训练后 AfterTrain")

# 提取 rouge-l F1 进行直观对比（百分制）
rl_before = scores_before["rouge-l"]["f"] * 100   # 训练前 ROUGE-L F1；float
rl_after  = scores_after["rouge-l"]["f"]  * 100   # 训练后 ROUGE-L F1；float
delta     = rl_after - rl_before                  # 微调带来的绝对提升；float

print(f"\n{'='*55}")
print(f"  ROUGE-L 对比（过滤 <think> 后）")
print(f"{'='*55}")
print(f"  训练前 ROUGE-L = {rl_before:.4f}")       # 基线分数
print(f"  训练后 ROUGE-L = {rl_after:.4f}")        # 微调后分数
print(f"  提升幅度 Δ    = {delta:+.4f}")           # 正值表示提升，负值表示下降


  过滤 <think> 后的 ROUGE 评估结果 — 训练前 BeforeTrain
  ROUGE-1     F1=20.1533  P=15.3897  R=32.7851
  ROUGE-2     F1=2.6225  P=1.9656  R=4.7426
  ROUGE-L     F1=9.7911  P=5.9260  R=32.3772

  过滤 <think> 后的 ROUGE 评估结果 — 训练后 AfterTrain
  ROUGE-1     F1=32.5849  P=34.6852  R=31.7861
  ROUGE-2     F1=8.0932  P=8.6521  R=7.9028
  ROUGE-L     F1=25.7173  P=27.5123  R=25.1474

  ROUGE-L 对比（过滤 <think> 后）
  训练前 ROUGE-L = 9.7911
  训练后 ROUGE-L = 25.7173
  提升幅度 Δ    = +15.9261
